# Module 2: When to Use GPUs (The Decision Framework)

Welcome! Today we will learn a structured framework to decide whether to migrate a pipeline stage to a GPU.

**Section Goals:**
* Learn the critical difference between low and high arithmetic intensity.
* Observe how low intensity calculations are slowed down by GPU copy overhead.
* Run benchmarking experiments for both scenarios.

### The Decision Framework

GPUs are expensive and data copying has a heavy PCIe latency tax. Do not use a GPU blindly. Use this checklist:

1. **Is the data size large?** (If size &lt; 100MB, CPU is faster because it avoids PCIe overhead).
2. **Is the operation compute-heavy?** (Arithmetic intensity = ratio of math steps to read steps).
3. **What is the data type?** (Floating-point math is optimal on GPU; string operations and regex are highly CPU-dependent).

### Visualizing the Decision Tree

Here is the data engineering flowchart to evaluate GPU migration:

![Decision Tree](images/decision-tree.svg)

### Step 1: Low Arithmetic Intensity Experiment

Let's measure the time taken to run a very simple operation (adding 1.0) on a CPU tensor vs. transferring the data to GPU, adding 1.0, and copying it back.

In [ ]:
import time
import torch

x = torch.randn(5000000, device="cpu")
t0 = time.perf_counter()
y = x + 1.0  # CPU math
print(f"CPU Time: {time.perf_counter() - t0:.4f}s")

In [ ]:
t0 = time.perf_counter()
x_gpu = x.to("cuda")  # Copy to GPU
y_gpu = x_gpu + 1.0  # Math
y_cpu = y_gpu.to("cpu")  # Copy back
print(f"GPU (with copy): {time.perf_counter() - t0:.4f}s")

### Step 2: High Arithmetic Intensity Experiment

Now let's run a compute-heavy operation (computing the matrix sine and cosine multiple times).

In [ ]:
t0 = time.perf_counter()
y = torch.sin(x).cos().sin()  # Heavy CPU math
print(f"CPU Heavy: {time.perf_counter() - t0:.4f}s")

In [ ]:
t0 = time.perf_counter()
x_gpu = x.to("cuda")
y_gpu = torch.sin(x_gpu).cos().sin()  # Heavy GPU math
y_cpu = y_gpu.to("cpu")
print(f"GPU Heavy (with copy): {time.perf_counter() - t0:.4f}s")

### Interpretation

For the simple addition, the CPU wins because transferring data over the PCIe bus takes longer than the math itself. For the heavy trigonometric calculation, the GPU wins easily because its parallel compute cores outperform the CPU enough to pay the PCIe tax.

### Module 2 Recap

* **Low arithmetic intensity** operations are best left on the CPU to avoid PCIe transfer costs.
* **High arithmetic intensity** operations justify the transfer overhead.
* Always evaluate the data type and operation complexity before migrating pipelines.